# authoring/02 — Clone an existing agent version and edit the draft

## What this demonstrates

The clone-and-edit workflow — the right way to make a governed change to a promoted agent. Instead of mutating the existing champion (which would break the audit trail), Verity lets you:

    GET   /api/v1/agents/{name}/config?version_id=<champion>
    POST  /api/v1/agents/{name}/versions/{champion_id}/clone
      → new draft with copied prompts + tools + delegations
          plus cloned_from_version_id pointing back at the source
    PATCH /api/v1/agents/{name}/versions/{draft_id}
      → edit the draft in place (only valid while lifecycle_state='draft')

The immutability contract: any version that has moved past `draft` (`candidate` / `staging` / `shadow` / `challenger` / `champion` / `deprecated`) is permanently frozen. PATCH on anything else returns `409 Conflict` with a message telling you to clone instead.

**Verity capabilities exercised**

- Transactional clone: one call produces the new version row   and all its prompt/tool/delegation associations.
- Clone provenance via the `cloned_from_version_id` column   — visible in the lineage graph below.
- Draft-only PATCH (immutability for promoted versions).
- Draft delete for clean rollback of the demo.


## Prerequisites

- `ds_workbench` application registered (`00_setup.ipynb`).
- `triage_agent` has a champion version (default seed).


In [ ]:
import os, sys
HERE = os.getcwd()
if os.path.basename(HERE) != 'ds_workbench':
    for candidate in (os.path.dirname(HERE),
                      os.path.dirname(os.path.dirname(HERE)),
                      '/home/jovyan/work'):
        if os.path.isdir(os.path.join(candidate, 'utility')):
            sys.path.insert(0, candidate); break

from utility.verity import VerityAPI, VerityAPIError
from utility.html import inject_style, badge, render_list, render_detail, render_cards

inject_style()
v = VerityAPI(application='ds_workbench')

In [ ]:
# Find the current champion version of triage_agent and its
# lineage context.
versions = v.list_agent_versions('triage_agent')
champion = next((ver for ver in versions if ver['lifecycle_state'] == 'champion'), None)
assert champion is not None, 'triage_agent has no champion version.'
CHAMPION_ID = champion['id']
print(f"champion: v{champion['version_label']}  id={CHAMPION_ID}")

# Pick a version label that isn't already taken.
taken = {vr['version_label'] for vr in versions}
DRAFT_LABEL = next(
    f"99.{minor}.0"
    for minor in range(100)
    if f"99.{minor}.0" not in taken
)
print(f"will clone into draft v{DRAFT_LABEL}")

## Execute — step 1: clone the champion into a new draft

One API call. Copies the version row + every prompt assignment + every tool authorization + every delegation row onto the new draft, and sets `cloned_from_version_id` so the provenance is visible.


In [ ]:
clone = v.clone_agent_version(
    name='triage_agent',
    source_version_id=CHAMPION_ID,
    new_version_label=DRAFT_LABEL,
    change_summary='authoring/02 clone demo — tweak temperature',
    developer_name='ds_workbench',
)
DRAFT_ID = clone['id']
print(f"draft v{clone['version_label']}  id={DRAFT_ID}")

## Execute — step 2: PATCH the draft

Modify something mutable on the draft. Here we swap the inference_config — typical reason for a clone is to try a cheaper or faster config against the existing prompt and tool set. Any field omitted from the PATCH body is left untouched (SQL COALESCE pattern).


In [ ]:
# Grab another inference_config to swap to — anything
# different from the current one.
configs = v.call('list_inference_configs')
current_cfg_id = champion['inference_config_id']
alt = next((c for c in configs if c['id'] != current_cfg_id), None)
assert alt is not None, 'Need at least 2 inference_configs registered.'
print(f"swapping inference_config: {alt['name']} ({alt['model_name']})")

patched = v.call(
    'update_agent_version',
    path_params={'name': 'triage_agent', 'version_id': DRAFT_ID},
    json={
        'inference_config_id': alt['id'],
        'change_summary': f"swap inference_config to {alt['name']}",
    },
)
print(f"PATCH result: {patched}")

## Execute — demonstrate the immutability guard

A PATCH against the champion (or anything non-draft) returns 409 Conflict. That's the contract that makes decision-log replay meaningful: past decisions always refer to a stable configuration.


In [ ]:
try:
    v.call(
        'update_agent_version',
        path_params={'name': 'triage_agent', 'version_id': CHAMPION_ID},
        json={'change_summary': 'this should fail'},
    )
    print('unexpected: PATCH on champion succeeded!')
except VerityAPIError as exc:
    print(f"expected {exc.status} — detail: {exc.detail}")

## Review results

The version lineage graph shows the new draft hanging off the champion via the clone edge (dashed purple). The resolved config confirms our PATCH landed on the draft — its inference config is the one we swapped in, while the champion still uses the original.


In [ ]:
from utility.visualizations import version_lineage_graph
version_lineage_graph(v.list_agent_versions('triage_agent'))

In [ ]:
draft_config = v.get_agent_config('triage_agent', version_id=DRAFT_ID)
champ_config = v.get_agent_config('triage_agent', version_id=CHAMPION_ID)

render_list(
    [
        {'field':'version_label',
         'champion': champ_config.get('version_label'),
         'draft':    draft_config.get('version_label')},
        {'field':'lifecycle_state',
         'champion': champ_config.get('lifecycle_state'),
         'draft':    draft_config.get('lifecycle_state')},
        {'field':'inference_config.name',
         'champion': champ_config['inference_config'].get('name'),
         'draft':    draft_config['inference_config'].get('name')},
        {'field':'inference_config.model',
         'champion': champ_config['inference_config'].get('model_name'),
         'draft':    draft_config['inference_config'].get('model_name')},
        {'field':'prompts.count',
         'champion': len(champ_config.get('prompts') or []),
         'draft':    len(draft_config.get('prompts') or [])},
        {'field':'tools.count',
         'champion': len(champ_config.get('tools') or []),
         'draft':    len(draft_config.get('tools') or [])},
    ],
    columns=[
        ('field','Field'),
        ('champion','Champion'),
        ('draft','Draft (ours)'),
    ],
    title='Champion vs draft — what changed',
    description='Clone copied prompts and tools; only the inference_config differs.',
)

## Cleanup

Remove the draft so the notebook is safe to re-run and the database ends up exactly where it started.


In [ ]:
try:
    v.call('delete_agent_version',
           path_params={'name': 'triage_agent', 'version_id': DRAFT_ID})
    print(f'draft v{DRAFT_LABEL} deleted')
except VerityAPIError as exc:
    print(f'draft delete failed: {exc.detail}')

---

To see the draft promoted (instead of deleted) into `candidate` and through the rest of the 7-state lifecycle, use the promote / rollback endpoints covered in `lifecycle/01_promote_and_rollback.ipynb`.
